# Data and preprocessing

This section discusses the catalog fields that are taken from each simulation and how these fields are used in the pipeline. The analysis uses FLAMINGO L1_m8 at z=0 and TNG300-1 at snapshot 99. To ensure a fair comparison between the simulations, we recompute stellar properties using comparable definitions.

The main notebooks use the included result tables and figures. The pickle files are local intermediate products and are not included in Git. There is currently no public release URL for the exact derived pickle files or trained checkpoints. The official links below provide the original simulation data; they do not provide this thesis's generated files.


## Start with the completed results

Follow the main notebook sequence in the [README](../README.md). Reading the results, plotting the saved tables, and following the bootstrap and common-support comparisons do not require raw catalogue downloads or another Roihu run. Optional local artifacts are separate from the public Git files.

Reconstructing catalogue caches produces the inputs used by the analysis. It does not recreate trained NPE checkpoints, saved posterior draws or bootstrap outcomes. Those require the exact saved artifact bundle or a new scientific run. Do not rerun the completed fits merely to read the repository.


## Obtain the original catalogues

**FLAMINGO.** Use the [official public data page](https://flamingo.strw.leidenuniv.nl/data.html). The existing helper accesses `FLAMINGO/L1_m8/L1_m8/SOAP-HBT/halo_properties_0078.hdf5` using `hdfstream.open('cosma', '/')`. Install [hdfstream](https://hdfstream-python.readthedocs.io/en/latest/); its [default `cosma` alias](https://hdfstream-python.readthedocs.io/en/latest/aliases.html) points to `https://dataweb.cosma.dur.ac.uk:8443/hdfstream`. The full parent population is needed for isolation checks, so a small catalogue slice does not reproduce the final selection.

**TNG300.** Register and log in on the [TNG300-1 download page](https://www.tng-project.org/data/downloads/TNG300-1/). Download the **snapshot 99 FoF and Subfind group catalogues**, preserving `groups_099/fof_subhalo_tab_099.*.hdf5` under a local `outputs` directory. Install the [official Illustris Python helpers](https://www.tng-project.org/data/docs/scripts/). Obtain a personal API key using the [TNG API instructions](https://www.tng-project.org/data/docs/api/). The later particle-feature scripts request cutouts only for the required final-pair members; a full particle snapshot is unnecessary.

Set `TNG_BASE` to the TNG300-1 `outputs` directory before starting Jupyter. Set `TNG_API_KEY` privately in the current environment, or use the private prompt in the optional notebook. Keep the key out of notebook source, saved outputs and Git. The scripts send it in the request header.

```powershell
$env:TNG_BASE = 'E:\simulation-data\TNG300-1\outputs'
```

Install the additional input-generation dependencies from `requirements-data.txt` and follow the official helper-package instructions. The original Roihu training pins are recorded separately in `requirements-research.txt`.


## Generate the cache files

Open [90_generate_catalogue_caches.ipynb](../notebooks/90_generate_catalogue_caches.ipynb). Every generation stage is disabled by default. Set `REBUILD=True` and add the stages you intend to run to `STAGES`. The order is:

1. `flamingo`: call the existing FLAMINGO pipeline and save the returned sample/pair dictionaries.
2. `tng`: call the existing `AnalogueSample` class and export the corrected pair/sample arrays.
3. `particles`: run the three existing resumable TNG particle-feature scripts.
4. `metallicity`: construct the shared aligned metallicity dex cache.
5. `manifest`: record the reconstructed version, verify counts and check feature alignment.
6. `sidecars`: run the existing bound-subhalo mass exporter.
7. `validate`: load the common pair tables and inspect the resulting populations.

Source helpers are preserved under `research/analogues/` and `research/TNG300/analogues/`. Generated inputs are written under `research/analogues/cache/`, `research/analogues/tng_for_comparison_v3/` and `research/data/derived/`, which are local artifact paths. The notebook sets `LG_PROJECT` to `research/` so that the original helpers keep their expected relative layout. Retain existing files and use a new cache version when inputs differ.

The old `TNG300/rebuild_comparison.py` is a migration that requires older saved arrays and feature caches. It cannot generate a fresh input set from only a raw catalogue. The optional notebook instead reuses the same `AnalogueSample` class directly. It preserves the original export's checked correction of the historical internal `log_tot_virial_mass` conversion and validates physical host masses.

The FLAMINGO pipeline command-line `--use-config` path prints the final pair count but does not save the returned dictionaries. The optional notebook explicitly writes `flamingo_sample.pkl` and `flamingo_pairs.pkl`. Saving the large parent `fields` dictionary is optional and off by default.


## Expected local inputs

Paths below are relative to `research/`. Original byte sizes indicate the scale of the inputs; newly serialized products may differ in bytes even when their arrays agree.

| File or group | Original size | Role |
|---|---:|---|
| `analogues/cache/run_001_mstar2e10/flamingo_sample.pkl` | 346.8 MB | FLAMINGO selected galaxy fields, IDs and hosts |
| `analogues/cache/run_001_mstar2e10/flamingo_pairs.pkl` | 10.0 MB | Pair indices, kinematics, masses and host flags |
| `analogues/cache/tng_features_v3/metallicity_log_zscore.pkl` | 27.7 MB | Aligned metallicity dex arrays and descriptive scalers |
| `analogues/cache/tng_features_v3/tng_sigma_star.pkl` | 69.9 kB | Global TNG subhalo ID to matched stellar dispersion |
| `analogues/cache/tng_features_v3/tng_kappa_corot.pkl` | 48.8 kB | Global TNG subhalo ID to matched co-rotation fraction |
| `analogues/tng_for_comparison_v3/*.npy` | About 2.1 MB | TNG selected galaxies and pair-aligned values |
| `analogues/tng_for_comparison_v3/selection_manifest.json` | Small JSON | Counts, units, version and array checksum |
| `data/derived/bound_subhalo_masses/flamingo_bound_subhalo_masses_v1.npz` and `.json` | 2.04 MB | Pair endpoint total bound masses and provenance |
| `data/derived/bound_subhalo_masses/tng_bound_subhalo_masses_v1.npz` and `.json` | 55.6 kB | Corresponding TNG masses and provenance |

The common pair-table loader reads these exact TNG arrays: `keep_idx`, `pairs_i`, `pairs_j`, `grnr`, `g_r`, `stellar_mass`, `separation`, `vr`, `vt`, `log_mass_ratio`, `log_m200c`, `have_same_host`. The broader comparison notebooks also use `metallicity` and `vel_disp`; the export retains `is_blue_blue` and `is_red_red` as descriptive flags.

`keep_idx` and the galaxy features are sample-aligned. Pair indices point into this sample. Translate to global subhalo IDs with `keep_idx[pairs_i]` and `keep_idx[pairs_j]`; using the pair indices directly in the TNG API would request the wrong galaxies.

The original FLAMINGO parent `flamingo_fields.pkl` is 8.63 GB and is unnecessary for ordinary cached pair-frame analysis. Keep it local if needed for parent-population checks. The m9 fiducial and jet caches are separate resolution/physics diagnostics and must not replace the m8 main sample.


## Definitions and checks

The final selected samples contain 142,713 FLAMINGO pairs and 1,530 TNG300 pairs. TNG has 1,253 same-host pairs and 277 different-host pairs. Original selected galaxy counts are 3,366,610 and 45,392, respectively. Counts are useful first checks, but they do not establish exact scientific input identity.

FLAMINGO SOAP masses are multiplied by `1e10` without division by h. TNG native masses are multiplied by `1e10/h`. Positions are physical kpc at z=0. TNG's physical box side is `205/h` Mpc, with `h=0.6774`. Pair radial velocities are peculiar velocities in both simulations.

The matched TNG stellar scripts exclude wind particles and require at least 100 valid stars. They use all bound stellar particles without an aperture cut. Stellar velocity dispersion subtracts the mass-weighted stellar centre-of-mass velocity and uses `sqrt(trace/3)`. Co-rotation uses the stellar angular-momentum direction. Metallicity uses the mass-weighted stellar metal fraction.

Both simulations use metallicity `log10(Z/0.0127)`. The cached per-simulation z-scores are descriptive quantities; NPE uses dex features and a scaler fitted on the training rows. The TNG-specific wind and particle-count cuts should not be described as identical requirements of the SOAP metallicity calculation.

Same-host pairs use a single host's M200c, whilst different-host pairs sum the independent M200c values. The alternative total-bound-subhalo sum is a separate target. The host third-member veto uses stellar mass. The spatial intruder veto uses total bound mass and includes potential intruders outside the stellar-selected sample.

A newly reconstructed input receives its own manifest and checksum. Before combining it with the fixed final splits or checkpoints, compare the exact pair-frame identity over:

```text
pair_index, id_a, id_b, host_i, host_j, same_host,
separation, vr, vt, mstar_a, mstar_b, log_m200c
```

The archived final identities are FLAMINGO `a23f726a1869af695bde0f1c62527ec595f5e1a59522e0f3b55b517b8811e69a` and TNG `6394a47d6f81aed87f0fdc0235ddd2d94bd1f1fc35a9669f93803d85bf75ffb2`. The fixed split-file SHA256 is `69a09e48f0abec5aef8ad9ecb70b877f6fbaa50c0ca85afbe2af7e78c9f2df0d`. Preserve the original connected-host-component partitions. Do not replace them with a random row split or round catalogue values to force a match.

The particle scripts resume from saved dictionary entries. A cached nonfinite entry is not automatically retried, so inspect the notebook's finite-feature checks before treating reconstruction as complete. Retain the cache and record a new version for any retry or correction.


## Saved results and the future artifact release

The original final-six, stellar-offset and observed-Local-Group runs are complete. Their saved summaries and draws allow reanalysis without retraining. The stellar-offset experiment uses its own historical stellar-sum/individual checkpoints; it is not an input shift of the final-six checkpoint. The later classifier-mixture posterior tables and Gaussian parameters are a separate completed diagnostic extension.

The fixed final archive alone does not contain the original catalogue inputs. A full saved-artifact release should include its inventory, SHA256 checksums, fixed split/calibration files, relevant checkpoint manifests and any posterior draws used for optional reanalysis. Add a verified DOI or release URL only after that bundle exists. The pickle files, raw catalogues and local artifact caches remain outside Git.
